In [19]:
#!/usr/bin/env python3
# -*- coding: UTF-8 -*-
"""
Video Crawler - JavHoo Scraper
===============================
Improved Python crawler that scrapes video metadata from javhoo.com.
"""

import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import re
import argparse
import sys
import os
import csv
from datetime import datetime

# ---------------------------------------------------------------------------
# Default configuration
# ---------------------------------------------------------------------------
DEFAULT_LIST_URL = "https://www.javhoo.com/ja/uncensored/page/"
DEFAULT_DETAIL_URL = "https://www.javhoo.com/ja/"
DEFAULT_DB_PATH = os.path.join(os.getcwd(), "video_crawler.db")
DEFAULT_DELAY = 1.0          # seconds between requests
DEFAULT_TIMEOUT = 20         # seconds for HTTP timeout
DEFAULT_RETRIES = 3          # retry attempts on failure
DEFAULT_USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/114.0.0.0 Safari/537.36"
)

LABEL_MAP = {
    "date":     ["發行日期", "発売日", "Release Date"],
    "producer": ["製作商",   "メーカー", "Maker", "Studio"],
    "series":   ["系列",     "シリーズ", "Series"],
    "cast":     ["演員",     "出演者",   "Cast", "Actress"],
    "duration": ["長度",     "収録時間", "Duration"],
}

ALL_LABELS = [
    "發行日期", "製作商", "系列", "演員", "長度", "類別", "標籤", "監督",
    "レーベル", "メーカー", "シリーズ", "出演者", "収録時間", "品番",
    "ジャンル", "タグ",
]

class VideoCrawler:
    def __init__(self, list_url, detail_url, db_path, delay, timeout, retries):
        self.list_url = list_url
        self.detail_url = detail_url
        self.db_path = db_path
        self.delay = delay
        self.timeout = timeout
        self.retries = retries
        self.headers = {
            "User-Agent": DEFAULT_USER_AGENT,
            "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
            "Referer": list_url.rstrip("0123456789"),
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)
        self._init_db()

    def _init_db(self):
        conn = sqlite3.connect(self.db_path)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS videos (
                ufanhao   TEXT PRIMARY KEY,
                title     TEXT DEFAULT '',
                imgurl    TEXT DEFAULT '',
                udate     TEXT DEFAULT '',
                date      TEXT DEFAULT '',
                producer  TEXT DEFAULT '',
                series    TEXT DEFAULT '',
                "cast"    TEXT DEFAULT '',
                duration  TEXT DEFAULT '',
                created_at TEXT DEFAULT (datetime('now')),
                updated_at TEXT DEFAULT (datetime('now'))
            )
        """)
        cursor = conn.execute("PRAGMA table_info(videos)")
        columns = [info[1] for info in cursor.fetchall()]
        if 'udate' not in columns:
            print("    ⚙ Migrating database: adding missing 'udate' column")
            conn.execute("ALTER TABLE videos ADD COLUMN udate TEXT DEFAULT ''")
        conn.commit()
        conn.close()

    def _upsert(self, conn, row: dict):
        conn.execute("""
            INSERT INTO videos (ufanhao, title, imgurl, udate, date, producer, series, "cast", duration, updated_at)
            VALUES (:ufanhao, :title, :imgurl, :udate, :date, :producer, :series, :cast, :duration, datetime('now'))
            ON CONFLICT(ufanhao) DO UPDATE SET
                title     = COALESCE(NULLIF(:title, ''),     videos.title),
                imgurl    = COALESCE(NULLIF(:imgurl, ''),    videos.imgurl),
                udate     = COALESCE(NULLIF(:udate, ''),    videos.udate),
                date      = COALESCE(NULLIF(:date, ''),      videos.date),
                producer  = COALESCE(NULLIF(:producer, ''),  videos.producer),
                series    = COALESCE(NULLIF(:series, ''),    videos.series),
                "cast"    = COALESCE(NULLIF(:cast, ''),      "cast"),
                duration  = COALESCE(NULLIF(:duration, ''),  videos.duration),
                updated_at = datetime('now')
        """, row)
        conn.commit()

    def _fetch(self, url: str) -> BeautifulSoup | None:
        for attempt in range(1, self.retries + 1):
            try:
                resp = self.session.get(url, timeout=self.timeout)
                if resp.status_code == 200: return BeautifulSoup(resp.text, "lxml")
                print(f"    ⚠ HTTP {resp.status_code} on attempt {attempt}/{self.retries}")
            except Exception as e:
                print(f"    ⚠ Error: {e}")
            if attempt < self.retries: time.sleep(self.delay * attempt * 2)
        return None

    def crawl_list_page(self, page: int) -> list[dict]:
        url = f"{self.list_url}{page}"
        print(f"📋 Fetching list page {page}...")
        soup = self._fetch(url)
        if not soup: return []
        articles = soup.find_all("article", class_=re.compile(r"excerpt"))
        items = []
        for art in articles:
            link_tag = art.find("a", href=True)
            if not link_tag: continue
            ufanhao = link_tag["href"].rstrip("/").split("/")[-1]
            h2 = art.find("h2"); title = h2.get_text(strip=True) if h2 else ""
            img_tag = art.find("img")
            imgurl = img_tag.get("data-src") or img_tag.get("src") or "" if img_tag else ""
            date_match = art.find(string=re.compile(r"\d{4}[-/]\d{2}[-/]\d{2}"))
            items.append({"ufanhao": ufanhao, "title": title, "imgurl": imgurl, "udate": date_match.strip() if date_match else "", "date": "", "producer": "", "series": "", "cast": "", "duration": ""})
        return items

    def crawl_detail(self, item: dict) -> dict:
        url = f"{self.detail_url}{item['ufanhao']}"
        soup = self._fetch(url)
        if not soup: return item
        content_text = soup.get_text(" ", strip=True)
        for field, labels in LABEL_MAP.items():
            if field == 'udate' or item.get(field): continue
            for label in labels:
                value = self._regex_extract(content_text, label)
                if value: item[field] = self._clean_value(value); break
        return item

    @staticmethod
    def _regex_extract(text: str, label: str) -> str:
        lookahead = "|".join(re.escape(lb) for lb in ALL_LABELS + ["識別碼", "類別", "標籤", "磁力"])
        pattern = rf"{re.escape(label)}\s*[:：]\s*(.+?)(?:\s*(?:{lookahead})\s*[:：]|磁力)"
        match = re.search(pattern, text)
        return match.group(1).strip() if match else ""

    @staticmethod
    def _clean_value(val: str) -> str:
        val = re.sub(r'[].*$', '', val)
        val = re.sub(r'赞\(.*?\)', '', val)
        val = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', val)
        return re.sub(r'\s+', ' ', val).strip()

    def fill_missing(self):
        conn = sqlite3.connect(self.db_path)
        missing = conn.execute('SELECT ufanhao, imgurl FROM videos WHERE producer="" OR "cast"=""').fetchall()
        print(f"🔍 Filling details for {len(missing)} records...")
        for uf, img in missing:
            detail = self.crawl_detail({"ufanhao": uf, "imgurl": img, "title": "", "udate": "", "date": "", "producer": "", "series": "", "cast": "", "duration": ""})
            self._upsert(conn, detail)
            time.sleep(self.delay)
        conn.close()

    def run(self, start_page: int, end_page: int, skip_details: bool = False):
        conn = sqlite3.connect(self.db_path)
        for page in range(start_page, end_page + 1):
            items = self.crawl_list_page(page)
            for item in items:
                if not skip_details: item = self.crawl_detail(item)
                self._upsert(conn, item)
            print(f"    ✅ Page {page} saved.")
        conn.close()

    @staticmethod
    def print_stats(db_path: str):
        conn = sqlite3.connect(db_path)
        total = conn.execute("SELECT COUNT(*) FROM videos").fetchone()[0]
        print(f"📊 Total records: {total}")
        conn.close()

    @staticmethod
    def export_csv(db_path, out_path):
        conn = sqlite3.connect(db_path)
        rows = conn.execute('SELECT ufanhao, title, imgurl, udate, producer, "cast" FROM videos').fetchall()
        with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow(["番号", "タイトル", "画像", "更新日", "製作商", "演員"])
            writer.writerows(rows)
        print(f"✅ Exported to {out_path}")
        conn.close()

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--pages", default="1")
    parser.add_argument("--fill", action="store_true")
    parser.add_argument("--export", help="Filename to export CSV")
    args, _ = parser.parse_known_args(sys.argv[1:] if len(sys.argv) > 1 else [])

    crawler = VideoCrawler(DEFAULT_LIST_URL, DEFAULT_DETAIL_URL, DEFAULT_DB_PATH, DEFAULT_DELAY, DEFAULT_TIMEOUT, DEFAULT_RETRIES)

    if args.fill: crawler.fill_missing()
    elif args.export: VideoCrawler.export_csv(DEFAULT_DB_PATH, args.export)
    else:
        try:
            s, e = map(int, args.pages.split('-')) if '-' in args.pages else (int(args.pages), int(args.pages))
            e = 1529
            crawler.run(s, e)
        except: crawler.run(1, 1)

    VideoCrawler.print_stats(DEFAULT_DB_PATH)

if __name__ == "__main__":
    main()

📋 Fetching list page 1...
    ✅ Page 1 saved.
📋 Fetching list page 2...
    ✅ Page 2 saved.
📋 Fetching list page 3...
    ✅ Page 3 saved.
📋 Fetching list page 4...
    ✅ Page 4 saved.
📋 Fetching list page 5...
    ✅ Page 5 saved.
📋 Fetching list page 6...
    ✅ Page 6 saved.
📋 Fetching list page 7...
    ✅ Page 7 saved.
📋 Fetching list page 8...
    ✅ Page 8 saved.
📋 Fetching list page 9...
    ✅ Page 9 saved.
📋 Fetching list page 10...
    ✅ Page 10 saved.
📋 Fetching list page 11...
    ✅ Page 11 saved.
📋 Fetching list page 12...
    ✅ Page 12 saved.
📋 Fetching list page 13...
    ✅ Page 13 saved.
📋 Fetching list page 14...
    ✅ Page 14 saved.
📋 Fetching list page 15...
    ✅ Page 15 saved.
📋 Fetching list page 16...
    ✅ Page 16 saved.
📋 Fetching list page 17...
    ✅ Page 17 saved.
📋 Fetching list page 18...
    ✅ Page 18 saved.
📋 Fetching list page 19...
    ✅ Page 19 saved.
📋 Fetching list page 20...
    ✅ Page 20 saved.
📋 Fetching list page 21...
    ✅ Page 21 saved.
📋 Fetching

# 新段落